# SFT LoRA: Mistral-7B-Instruct-v0.3 

This notebook fine-tunes `mistralai/Mistral-7B-Instruct-v0.3` with LoRA using:

`instruction + report -> JSON`


In [1]:
# environment

import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_SILENT"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("VRAM GB:", round(props.total_memory / 1024**3, 1))

CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM GB: 95.0


In [3]:
from datasets import load_dataset

In [4]:
import json
import math
from pathlib import Path

from datasets import load_dataset
from transformers import AutoTokenizer


def validate_jsonl(path):
    """Validate that each JSONL row has the expected report/label schema."""
    path = Path(path)
    n = 0
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            row = json.loads(line)
            if not isinstance(row.get("report"), str) or not row["report"].strip():
                raise ValueError(f"{path}:{line_no} missing non-empty string field 'report'")
            if not isinstance(row.get("label"), dict):
                raise ValueError(f"{path}:{line_no} missing object field 'label'")
            n += 1
    print(f"OK: {path} contains {n:,} examples")
    return n


def summarize_token_lengths(path, model_id, max_examples=None):
    """Print simple token-length statistics for the raw report+label examples."""
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    lengths = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if max_examples is not None and i >= max_examples:
                break
            row = json.loads(line)
            text = row["report"] + "\n" + json.dumps(row["label"], ensure_ascii=False)
            lengths.append(len(tokenizer(text, add_special_tokens=True)["input_ids"]))
    if not lengths:
        print("No examples found")
        return {}
    stats = {
        "n": len(lengths),
        "min": min(lengths),
        "p50": sorted(lengths)[len(lengths) // 2],
        "p95": sorted(lengths)[math.floor(0.95 * (len(lengths) - 1))],
        "max": max(lengths),
    }
    print(stats)
    return stats


def preview_training_pair(path, n=1):
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            row = json.loads(line)
            print("REPORT:\n", row["report"][:1000])
            print("\nLABEL:\n", json.dumps(row["label"], ensure_ascii=False, indent=2))
            print("-" * 80)


# Prepre dataset
{"report":"Compte rendu anatomopathologique ...","label":{"echantillon":"tumorectomie","numero_gang_prel":14,"numero_gang_att":2,"taille":18,"diagnostic":"carcinome canalaire infiltrant","grade":"SBR2","emboles":"non","ki67":22,"re":"positifs","rp":"positifs","her2":"négatif","marges_saines":"oui","rupture_capsulaire":"non"}}

Categorical missing/ambiguous values -> "unknown"

Numeric missing/ambiguous values -> null

In [5]:
import ast

def flatten_entry_data(entry_data):
    if isinstance(entry_data, str):
        entry_data = ast.literal_eval(entry_data)

    return {
        att: info.get("value", "unknown")
        for att, info in entry_data.items()
    }

    
def normalize_examples_dataframe(data):
    """Return a dataframe with at least columns: entry_data, text.

    Accepted inputs:
    - pd.DataFrame with entry_data/text columns
    - pd.DataFrame with eval column containing list(s) of records
    - dict with key "eval" containing records
    - list of records
    """
    if isinstance(data, dict) and "eval" in data:
        return pd.DataFrame(data["eval"])

    if isinstance(data, list):
        return pd.DataFrame(data)

    if not isinstance(data, pd.DataFrame):
        raise TypeError("data must be a DataFrame, dict with 'eval', or list of records")

    if {"entry_data", "text"}.issubset(data.columns):
        return data.copy()

    if "eval" in data.columns:
        records = []
        for value in data["eval"].dropna():
            if isinstance(value, list):
                records.extend(value)
            elif isinstance(value, dict):
                records.append(value)
            else:
                raise TypeError(f"Unexpected eval cell type: {type(value)}")
        return pd.DataFrame(records)

    raise ValueError("Could not find columns entry_data/text or an eval column.")





In [6]:
import pandas as pd
import json

df_rk = pd.read_csv("../LLMEvaluation/selectedMixtral_RAG.csv")

normalized_df_rk_1_2 = normalize_examples_dataframe(df_rk)
normalized_df_rk_1_2["entry_data"] = normalized_df_rk_1_2["entry_data"].apply(flatten_entry_data)

normalized_df_rk_1_2.columns

Index(['index', 'text', 'entry_data', 'source'], dtype='object')

In [7]:
import pandas as pd
import json
import ast

def parse_entry_data(x):
    if isinstance(x, dict):
        return x

    if pd.isna(x):
        return {}

    if isinstance(x, str):
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return ast.literal_eval(x)

    return {}

In [8]:
normalized_df_rk_1_2["entry_data_parsed"] = normalized_df_rk_1_2["entry_data"].apply(parse_entry_data)

entry_df = pd.json_normalize(normalized_df_rk_1_2["entry_data_parsed"])
entry_df = entry_df.drop(
    columns=["re_perc", "rp_perc"],
    errors="ignore"
)

COLUMN_MAPPING = {
    "type_diagnostique": "echantillon",
    "ganglions_preleves": "numero_gang_prel",
    "ganglions_atteints": "numero_gang_att",
    "taille_tumor_0": "taille",
    "ref_morpho_name_tumor_0": "diagnostic",
    "ref_grade_tumor_0": "grade",
    "embols_vasculaires_tumor_0": "emboles",
    "ki67_tumor_0": "ki67",
    "re_tumor_0": "re",
    "rp_tumor_0": "rp",
    "cerb_tumor_0": "her2",
    "marges_saines_tumor_0": "marges_saines",
    "rupture_capsulaire": "rupture_capsulaire",
}


entry_df = entry_df.rename(columns=COLUMN_MAPPING)
entry_df["grade"] = (
    entry_df["grade"]
    .astype("string")
    .str.replace("MSBR", "SBR", regex=False)
)
entry_df.columns

Index(['echantillon', 'numero_gang_prel', 'numero_gang_att',
       'rupture_capsulaire', 'taille', 're', 'rp', 'emboles', 'her2',
       'marges_saines', 'ki67', 'diagnostic', 'grade'],
      dtype='object')

In [1]:
FIELD_ORDER = [
    "echantillon",
    "numero_gang_prel",
    "numero_gang_att",
    "taille",
    "diagnostic",
    "grade",
    "emboles",
    "ki67",
    "re",
    "rp",
    "her2",
    "marges_saines",
    "rupture_capsulaire",
]

entry_df = entry_df.reindex(columns=FIELD_ORDER)

entry_df.head()

In [2]:
flat_df = pd.concat(
    [
        normalized_df_rk_1_2[['index',"text"]].reset_index(drop=True),
        entry_df.reset_index(drop=True),
    ],
    axis=1,
)

flat_df.head()

In [12]:
NUMERIC_FIELDS = {
    "numero_gang_prel",
    "numero_gang_att",
    "taille",
    "ki67",
}

CATEGORICAL_FIELDS = set(FIELD_ORDER) - NUMERIC_FIELDS

def clean_numeric(value):
    if value is None or pd.isna(value) or value == "":
        return None

    if isinstance(value, str):
        value = value.strip()
        value = value.replace("%", "")
        value = value.replace(",", ".")
        if value == "":
            return None

    try:
        return int(float(value))
    except Exception:
        return None


def clean_categorical(value):
    if value is None or pd.isna(value) or value == "":
        return "unknown"

    value = str(value).strip()

    if value.lower() in ["nan", "none", "null", "na", "n/a"]:
        return "unknown"

    return value


def clean_label_row(row):
    label = {}

    for field in FIELD_ORDER:
        value = row.get(field)

        if field in NUMERIC_FIELDS:
            label[field] = clean_numeric(value)
        else:
            label[field] = clean_categorical(value)

    return label

### create finetuning jsonl

In [3]:
records = []
OUTPUTFOLDER = "training_data/sft_Mx_Rk"
Path(OUTPUTFOLDER).mkdir(parents=True, exist_ok=True)

for i, row in flat_df.iterrows():
    label = clean_label_row(entry_df.iloc[i])

    records.append({
        "report": str(row["text"]).strip(),
        "label": label,
    })

with open(OUTPUTFOLDER + "/all_data.jsonl", "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

validate_jsonl(OUTPUTFOLDER + "/all_data.jsonl")

with open(OUTPUTFOLDER + "/all_data.jsonl", "r", encoding="utf-8") as f:
    example = json.loads(next(f))

print(json.dumps(example, indent=2, ensure_ascii=False))


### split train_dev_val

In [14]:
import random

random.seed(42)

with open(OUTPUTFOLDER + "/all_data.jsonl", "r", encoding="utf-8") as f:
    items = [json.loads(line) for line in f]

random.shuffle(items)

n = len(items)
n_train = int(0.9 * n)
n_valid = int(0.1 * n)

train_items = items[:n_train]
valid_items = items[n_train:n_train+n_valid]

def write_jsonl(path, data):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

write_jsonl(OUTPUTFOLDER + "/train.jsonl", train_items)
write_jsonl(OUTPUTFOLDER + "/valid.jsonl", valid_items)


print(len(train_items), len(valid_items))


367 40


# Fine-tune Mistral-7B-Instruct-v0.3

Hugging Face Transformers + TRL + PEFT for QLoRA fine-tuning. 
Format each example with Mistral's instruct chat template and train only the LoRA adapter weights.


In [4]:
import os
import json
from pathlib import Path
from typing import Any, Dict, Optional

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
OUTPUT_DIR = "models/mistral-7b-instruct-v0.3-pathology-lora-rk"


MAX_SEQ_LENGTH = 2048

FIELD_ORDER = [
    "echantillon",
    "numero_gang_prel",
    "numero_gang_att",
    "taille",
    "diagnostic",
    "grade",
    "emboles",
    "ki67",
    "re",
    "rp",
    "her2",
    "marges_saines",
    "rupture_capsulaire",
]

NUMERIC_FIELDS = {"numero_gang_prel", "numero_gang_att", "taille", "ki67"}

SYSTEM_PROMPT = """
Tu es un système d'extraction d'information à partir de comptes rendus anatomopathologiques de cancer du sein.

Retourne uniquement un objet JSON valide.
N'invente aucune information.

Si une information est absente, ambiguë ou non déductible:
- utilise "unknown" pour les variables catégorielles
- utilise null pour les variables numériques.

Variables à extraire:
- echantillon: mammectomie, tumorectomie, curage ganglionnaire, biopsie, exérèse chirurgicale, unknown
- numero_gang_prel: nombre de ganglions prélevés/examinés, integer ou null
- numero_gang_att: nombre de ganglions atteints/envahis, integer ou null
- taille: taille tumorale maximale en mm, integer ou null
- diagnostic: histologie principale, string
- grade: SBR1, SBR2, SBR3, unknown
- emboles: oui, non, unknown
- ki67: pourcentage Ki67, integer ou null
- re: positifs, négatifs, unknown
- rp: positifs, négatifs, unknown
- her2: négatif, douteux, positif, unknown
- marges_saines: oui, non, unknown
- rupture_capsulaire: oui, non, unknown

Règles:
- Convertir les cm en mm pour taille.
- HER2 0 ou 1+ = négatif.
- HER2 2+ = douteux.
- HER2 3+ ou amplification FISH/CISH/SISH = positif.
- RE/RP positifs si expression positive, même faible.
- Marges R0/non envahies/saines = oui.
- Marges atteintes/R1/envahies = non.
""".strip()


def parse_label(label: Any) -> Dict[str, Any]:
    if isinstance(label, str):
        label = json.loads(label)
    if not isinstance(label, dict):
        raise ValueError(f"label must be a dict or JSON string, got {type(label)}")
    return label


def ordered_label(label: Dict[str, Any]) -> Dict[str, Any]:
    """Stable field order, with numeric missing values normalized to None."""
    out: Dict[str, Any] = {}
    for field in FIELD_ORDER:
        value = label.get(field, "unknown")
        if field in NUMERIC_FIELDS:
            if value is None or value == "" or str(value).lower() in {"unknown", "nan", "none", "null"}:
                value = None
            else:
                value = int(value)
        else:
            if value is None or value == "":
                value = "unknown"
            else:
                value = str(value)
        out[field] = value
    return out


def dumps_label(label: Dict[str, Any]) -> str:
    return json.dumps(
        ordered_label(parse_label(label)),
        ensure_ascii=False,
        separators=(",", ":"),
    )


def build_report_user_message(report: str) -> str:
    return f"""{SYSTEM_PROMPT}

Compte rendu:
\"\"\"
{report}
\"\"\"

Retourne uniquement le JSON."""


def build_extraction_messages(report: str, label: Optional[Dict[str, Any]] = None):
    """Zero-shot training/inference prompt. No few-shots, no artificial acknowledgement."""
    messages = [{"role": "user", "content": build_report_user_message(report)}]
    if label is not None:
        messages.append({"role": "assistant", "content": dumps_label(label)})
    return messages


def format_for_sft(example):
    messages = build_extraction_messages(example["report"], example["label"])
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    if tokenizer.eos_token and not text.endswith(tokenizer.eos_token):
        text += tokenizer.eos_token
    return {"text": text}


Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


TRAIN_JSONL = OUTPUTFOLDER + "/train.jsonl"
VALID_JSONL = OUTPUTFOLDER + "/valid.jsonl"
TEST_JSONL = OUTPUTFOLDER + "/test.jsonl"

validate_jsonl(TRAIN_JSONL)
validate_jsonl(VALID_JSONL)
if Path(TEST_JSONL).exists():
    validate_jsonl(TEST_JSONL)
    
raw = load_dataset(
    "json",
    data_files={
        "train": TRAIN_JSONL,
        "validation": VALID_JSONL,
        # "test": TEST_JSONL,
    },
)

dataset = raw.map(format_for_sft, remove_columns=raw["train"].column_names)
dataset


In [5]:
compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=compute_dtype,
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    num_train_epochs=2,
    learning_rate=1e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    optim="adamw_torch",
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved LoRA adapter to: {OUTPUT_DIR}")
